[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rl-tools/rl-tools/blob/master/interface/python/hyperdrone/hyperdrone/examples/Overview.ipynb)

# HyperDrone Overview

The environment in five cells: a content-addressed ProcTHOR scene, four drone models flying
in one instance, one simulation step, and every output modality the environment renders —
the shot of Figure 1 of the HyperDrone paper, whose API appendix is generated from this
notebook. `pip install hyperdrone` is the only prerequisite. The first construction of an
environment configuration JIT-compiles it; later constructions load it from the cache.

In [ ]:
import importlib.util
import site
import sysconfig
from pathlib import Path

checkout = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "interface/python/hyperdrone/pyproject.toml").is_file()), None)
if checkout is not None:
    %pip install -q -e "{checkout}/interface/python/hyperdrone[examples]"
    site.addsitedir(sysconfig.get_paths()["purelib"])  # activates the editable install's .pth without a kernel restart
elif importlib.util.find_spec("hyperdrone") is None:  # fresh runtime (e.g. Colab)
    %pip install -q "hyperdrone[examples]"

The folded cell below holds only the false-color helpers of Figure 1 (depth, normals,
label palette, flow), the mosaic layout and a pose formatter.

In [ ]:
from colorsys import hsv_to_rgb

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import hyperdrone
from hyperdrone import render

hyperdrone.set_verbosity("warning")
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pad_inches": 0}
FLOW_MAX_PIXELS = 25.0


def pose(position, yaw_degrees):
    half = np.deg2rad(yaw_degrees) / 2
    return [*position, float(np.cos(half)), 0.0, 0.0, float(np.sin(half))]


def to_uint8(image):
    return image if image.dtype == np.uint8 else (np.clip(image, 0, 1) * 255).astype(np.uint8)


def depth_image(depth):
    valid = np.isfinite(depth) & (depth > 0)
    image = np.zeros(depth.shape, dtype=np.uint8)
    if valid.any():
        minimum, maximum = depth[valid].min(), depth[valid].max()
        normalized = np.clip((depth[valid] - minimum) / (maximum - minimum + 1e-6), 0, 1)
        image[valid] = ((1 - normalized) * 255).astype(np.uint8)
    return np.repeat(image[..., None], 3, axis=-1)


def normals_image(normals):
    valid = np.isfinite(normals).all(axis=-1) & np.any(normals != 0, axis=-1)
    image = np.zeros(normals.shape, dtype=np.uint8)
    image[valid] = np.rint(np.clip((normals[valid] + 1) * 0.5, 0, 1) * 255)
    return image


def segmentation_image(labels, *, semantic=False):
    ids, inverse = np.unique(labels, return_inverse=True)
    palette = np.zeros((len(ids), 3), dtype=np.uint8)
    for index, label in enumerate(ids):
        if label == np.iinfo(np.uint32).max:
            continue  # a missed ray; instance ID zero is valid geometry
        if semantic and label == 0:
            palette[index] = 128  # undefined semantic class (the drone models carry none)
        else:
            hue = (int(label) * 0.618033988749895 + 0.08) % 1.0
            palette[index] = np.rint(np.array(hsv_to_rgb(hue, 0.65, 0.95)) * 255)
    return palette[inverse].reshape(*labels.shape, 3)


def flow_image(flow, status):
    valid = (status == render.FlowStatus.VALID) & np.isfinite(flow).all(axis=-1)
    vectors = flow[valid]
    hsv = np.zeros((*status.shape, 3), dtype=np.uint8)
    hsv[..., 0][valid] = (np.mod(np.arctan2(vectors[:, 1], vectors[:, 0]) / (2 * np.pi), 1) * 255).astype(np.uint8)
    hsv[..., 1][valid] = np.rint(np.clip(np.linalg.norm(vectors, axis=-1) / FLOW_MAX_PIXELS, 0, 1) * 255)
    hsv[..., 2][valid] = 255
    return np.array(Image.frombytes("HSV", (status.shape[1], status.shape[0]), hsv.tobytes()).convert("RGB"))


def mosaic(tiles, columns=3, dpi=200):
    tiles = {title: to_uint8(pixels)[..., :3] for title, pixels in tiles.items()}
    size = next(iter(tiles.values())).shape[0]
    gap, band = size // 128, size // 9  # column gap; label band above each row
    rows = -(-len(tiles) // columns)
    canvas = np.full((rows * (band + size), columns * size + (columns - 1) * gap, 3), 255, dtype=np.uint8)
    figure = plt.figure(figsize=(canvas.shape[1] / dpi, canvas.shape[0] / dpi), dpi=dpi)
    axis = figure.add_axes([0, 0, 1, 1])
    for index, (title, pixels) in enumerate(tiles.items()):
        row, column = divmod(index, columns)
        in_row = min(columns, len(tiles) - row * columns)
        y = row * (band + size) + band
        x = column * (size + gap) + (columns - in_row) * (size + gap) // 2  # center a short last row
        canvas[y:y + size, x:x + size] = pixels
        axis.text(x + size / 2, y - band / 2, title, ha="center", va="center", fontsize=20)
    axis.imshow(canvas, interpolation="none")
    axis.axis("off")
    plt.show()

## Scene and drone table

Scenes are GLB files addressed by content hash: the `Train200` catalog pins the 200 ProcTHOR
training scenes, and `conta` fetches a scene once into a shared cache. The drone table lists
the models that fly together in one instance, each a rigged GLB paired with the dynamics
model of the same name from the registry.

In [ ]:
import numpy as np
from hyperdrone import render
from hyperdrone.env import EnvConfig, MultiEnvironment
from hyperdrone.examples.data import rigged_model_path

scene = render.datasets.Train200()["ProcTHOR-Train-1"]
scenes = [scene.reference]
MODELS = ("crazyflie_brushless", "arpl", "x500", "savagebee_pusher")
drones = [{"asset": rigged_model_path(name),
           "parameters": {"dynamics": {"dynamics": {"model": name}}}}
          for name in MODELS]

## Environment

`EnvConfig` is the compile-time part of an environment and its JIT key: resolution,
outputs, shading, motion-blur samples and the agent layout. One instance groups four
mutually visible agents; agent 0 carries the only camera, and its mount puts that camera
at the pose of Figure 1. `semantic_segmentation=True` makes the `segmentation` output
report class IDs instead of instance IDs. Each model's dynamics declare its rotor speed,
so the propellers spin at their physical rate.

In [ ]:
config = EnvConfig(preset="x500_fpv", num_agents=4, num_camera_agents=1,
                   instances_per_environment=1, width=1024, height=1024,
                   fidelity="high", motion_blur_samples=16, dataset="procthor",
                   semantic_segmentation=True,
                   output=("rgb", "depth", "segmentation", "normals", "flow"))
mount = {"position": [-0.2102, 0.2639, 0.3870],  # Figure 1's camera in
         "orientation": [0.6276, 0.2865, 0.2584, -0.6763]}  # agent 0 body frame
drones[0]["parameters"]["cameras"] = [{"fov": 80, "mount": {"pose": mount}}]

## Initialization

Runtime parameters are a partial dict merged over the nominal values. Here the
initialization distribution is pinned to a level attitude with the propellers at full
speed and a random initial velocity of up to 6 m/s per axis, which is what the flow tile
will show.

In [ ]:
initial = {"guidance": 0, "max_position": 0, "max_angle": 0,
           "max_angular_velocity": 0, "max_linear_velocity": 6.0,
           "relative_rpm": True, "min_rpm": 1, "max_rpm": 1}
parameters = {"dynamics": {"mdp": {"init": initial}}}
env = MultiEnvironment(scenes, config=config, drones=drones, seed=0,
                       parameters=parameters)

## Reset and step

`reset` samples the scene placement and the poses; a recorded spawn point replays a
specific one, here the four poses of Figure 1 with the scene at the origin. `observe`
renders lazily, and `step` integrates 10 ms of dynamics for all agents. A policy would map
the observation to actions; here every agent applies its hover throttle from the resolved
drone table.

In [ ]:
poses = [pose((-1.602, -3.994, 1.145), 65), pose((-0.459, -4.226, 1.120), 30),
         pose((-0.365, -5.578, 1.073), -35), pose((-1.806, -4.461, 0.982), -20)]
shot = {"scene": scene.reference, "drone": [0, 1, 2, 3], "poses": poses,
        "scene_translation": [0, 0, 0], "scene_yaw": [1, 0],
        "shutter_fraction": 0.2}
table = [d["parameters"]["dynamics"]["dynamics"] for d in env.drones]
hover = np.repeat([2 * t["hovering_throttle_relative"] - 1 for t in table], 4)
actions = hover[None].astype(np.float32)  # what a policy would output
env.reset(spawn_points=[shot])
env.observe()                             # first capture: the flow reference
env.step(actions)                         # a policy would act here

## Outputs

Accessors return `(instances, views, H, W[, C])` snapshots and accept `out=` buffers
(CUDA arrays on an OptiX environment); `frames()` is the camera image behind the
observation. Flow is the displacement to the previous capture, valid where
`flow_status()` says so; the false-color mappings are those of Figure 1, with flow
saturating at 25 px.

In [ ]:
flow, status = env.flow()[0, 0], env.flow_status()[0, 0]
mosaic({
    "RGB": env.frames()[0],
    "Depth": depth_image(env.depth()[0, 0]),
    "Surface normals": normals_image(env.normals()[0, 0]),
    "Semantic segmentation": segmentation_image(env.segmentation()[0, 0],
                                                semantic=True),
    "Optical flow": flow_image(flow, status),
})
print(f"hyperdrone {hyperdrone.__version__} on {env.render_backend}, "
      f"{np.mean(status == render.FlowStatus.VALID):.1%} valid flow")

Flying the environment with a pilot, custom tasks and the C++ extension ladder are in
[HyperDrone.ipynb](HyperDrone.ipynb) and [Orbiter.ipynb](Orbiter.ipynb); the renderer and
the dynamics as standalone peers are in [Renderer.ipynb](Renderer.ipynb); throughput is
[Benchmark.ipynb](Benchmark.ipynb).